# conditional Edge

1. `state의 값을 건들이지 않으면 langgraph에서 node가 아님`
2. 현재 상황의 state 혹은 다른 기준을 바탕으로, 특정 값을 `return` 함
3. 해당 값과 매칭딘 node가 실행됨

### 예시

In [17]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [18]:
# State 정의

class SystemState(TypedDict):
    message : str
    is_ok : bool
    next_step : str


In [26]:
def process_message(state: SystemState):
    print('---메세지 처리 노드---', state)

    msg = state['message']
    if('error' in msg.lower()) or ('에러' in msg):
        result = False
    else:
        result = True

    return{'is_ok': result}
    

def normal_node(state: SystemState):
    print('---문제 없음 노드', state)
    return{'next_step': '문제 없음. 계속 진행'}

def error_node(state: SystemState):
    print('---에러 발생 노드---', state)
    return{'next_step': '에러 발생 비상'}

In [27]:
# Router (Node 아님 -> State를 건들지 않음 다음 값을)
def is_ok_router(state:SystemState):
    if state['is_ok']:
        return 'OK'
    else:
        return 'Error'

In [28]:
builder = StateGraph(SystemState)
#노드 등록
builder.add_node('메세지처리', process_message)
builder.add_node('일반 노드', normal_node)
builder.add_node('에러 노드',error_node)

#연결
builder.add_edge(START, '메세지처리')
#라우터에 따라 결정
builder.add_conditional_edges('메세지처리', is_ok_router,{
  'OK' : '일반 노드',
  'Error' : '에러 노드',
})
builder.add_edge('일반 노드', END)
builder.add_edge('에러 노드', END)

graph = builder.compile()

In [29]:
#graph.invoke({'message': '좋아좋아'})
graph.invoke({'message':'에러'})

---메세지 처리 노드--- {'message': '에러'}
---에러 발생 노드--- {'message': '에러', 'is_ok': False}


{'message': '에러', 'is_ok': False, 'next_step': '에러 발생 비상'}